# CAMELS: Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 14-04-2026<br>

**Introduction:**<br>
This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and exports a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

from ocab.config import Config
from ocab.plots import plot_station_timeseries, create_station_html


In [2]:

## Configuration

cfg = Config('config_CAMELS_v200.yml')

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_out = Path('../../docs/timeseries/stations')
path_plots = path_out / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# decimals in output timeseries
rounding = {
    'discharge_cms': 3,
    'discharge_mm': 1,
    'temp_degC': 1,
    'precip_mm': 1,
    'pet_mm': 1,
}

variables = {
    'ta_mean': 'temp_degC', 
    'pr_mean': 'precip_mm', 
    'e0_mean': 'pet_mm',
}


In [ ]:

## Create time series

# load stations
stations = gpd.read_file(cfg.path_gis / 'stations.geojson').set_index('id')

# process timeseries for each station
for ID in tqdm(stations.index, desc='stations'):

    # discharge timeseries
    try:
        dis = pd.read_parquet(path_in / 'discharge' / f'{ID:04d}.parquet')
        dis.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        dis['discharge_mm'] = dis['discharge_cms'] / stations.loc[ID, 'catch_skm'] * 86400 / 1000
    except Exception as e:
        print(f'Error loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # meteo timeseries
    try:
        meteo = pd.read_parquet(path_in / 'meteo' / f'{ID:04d}.parquet').loc[ID]
        meteo.rename(columns=variables, inplace=True, errors='ignore')
        # correct dates
        meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
        start = max(meteo.first_valid_index(), dis.first_valid_index())
        end = min(meteo.last_valid_index(), dis.last_valid_index())
        meteo = meteo.loc[start:end]
    except Exception as e:
        print(f'Error loading meteo timeseries for station {ID:04d}: {e}')
        continue

    # merge timeseries
    ts = pd.concat([dis, meteo], axis=1)
    ts = ts[ts.columns.intersection(rounding)].round(rounding)

    # export timeseries
    ts.to_parquet(path_out / f'{ID:04d}.parquet')

    # extract attributes and time series
    attrs = stations.loc[ID]

    # create time series plot
    try:
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title(), 
            attrs['river'].title(), 
            attrs['basin'].title()
        )
        fig = plot_station_timeseries(
            ts,
            area=attrs['catch_skm'],
            title=title,
            regime=attrs['regime'],
            save=True
        )

        # save plot as HTML
        create_station_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except:
        print(f"The plot for time series {ID} could'nt be created")

stations:   0%|          | 0/1009 [00:00<?, ?it/s]

Error loading meteo timeseries for station 1080: name 'resops' is not defined
Error loading meteo timeseries for station 1103: name 'resops' is not defined
Error loading meteo timeseries for station 1105: name 'resops' is not defined
Error loading meteo timeseries for station 1106: name 'resops' is not defined
Error loading meteo timeseries for station 1107: name 'resops' is not defined
Error loading meteo timeseries for station 1109: name 'resops' is not defined
Error loading meteo timeseries for station 1141: name 'resops' is not defined
Error loading meteo timeseries for station 1158: name 'resops' is not defined
Error loading meteo timeseries for station 1163: name 'resops' is not defined
Error loading meteo timeseries for station 1164: name 'resops' is not defined
Error loading meteo timeseries for station 1175: name 'resops' is not defined
Error loading meteo timeseries for station 1186: name 'resops' is not defined
Error loading meteo timeseries for station 1196: name 'resops' i

KeyboardInterrupt: 